In [6]:
import os
import sys
from pathlib import Path

import torch
from datasets import load_dataset

# Make sure we can import train1.py from this notebook directory.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

from train1 import GPTConfig, miniGPT

In [7]:
# Build a small GPT from scratch (tokenizer may be pretrained).
config = GPTConfig(
    block_size=256,
    n_layers=6,
    n_embd=384,
    n_heads=6,
    dropout=0.1,
)

trainer = miniGPT.from_tokenizer_name("gpt2", config=config)

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
trainer = trainer.to(device)
print(f"Using device: {device}")

Using device: mps


In [9]:
# Load huggingface/all-recipes and split exactly as requested.
raw = load_dataset("corbt/all-recipes")

# Some configs expose only a train split; this handles both cases safely.
if "train" in raw:
    base_split = raw["train"]
else:
    first_key = list(raw.keys())[0]
    base_split = raw[first_key]

split = base_split.train_test_split(test_size=0.05, seed=42)
train_full = split["train"]
test_set = split["test"]

print(f"train_full: {len(train_full):,} samples")
print(f"test_set: {len(test_set):,} samples")

Generating train split: 100%|██████████| 2147248/2147248 [00:01<00:00, 1644575.32 examples/s]


train_full: 2,039,885 samples
test_set: 107,363 samples


In [11]:
# Make a smaller training subset so experiments run faster.
subset_size = min(20000, len(train_full))
subset_seed = 42
train_subset = train_full.shuffle(seed=subset_seed).select(range(subset_size))

# Pick a likely text field from the dataset.
candidate_columns = [
    "text",
    "recipe",
    "instructions",
    "directions",
    "title",
    "ingredients",
    "source",
    "input",
]

available_columns = train_subset.column_names
text_column = next((c for c in candidate_columns if c in available_columns), None)
if text_column is None and len(available_columns) == 1:
    text_column = available_columns[0]
if text_column is None:
    raise ValueError(f"No expected text column found. Available: {available_columns}")

def _flatten_to_text(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return " ".join(_flatten_to_text(v) for v in value)
    if isinstance(value, dict):
        # Prefer common content keys when present.
        for key in ["text", "input", "recipe", "instructions", "directions", "title", "ingredients"]:
            if key in value:
                return _flatten_to_text(value[key])
        return " ".join(f"{k}: {_flatten_to_text(v)}" for k, v in value.items())
    return str(value)

def to_text(row):
    return _flatten_to_text(row.get(text_column)).strip()

train_texts = [to_text(r) for r in train_subset]
train_texts = [t for t in train_texts if t]

print(f"Selected text column: {text_column}")
print(f"train_subset used: {len(train_texts):,} samples")

Selected text column: input
train_subset used: 20,000 samples


In [12]:
# Train on the smaller subset.
trainer.train(
    train_texts,
    epochs=1,
    batch_size=8,
    learning_rate=3e-4,
    warmup_ratio=0.03,
    min_lr_ratio=0.1,
)

# Save your custom-trained model + tokenizer files.
output_dir = NOTEBOOK_DIR / "trained_model_small"
trainer.save(str(output_dir))
print(f"Saved to: {output_dir}")

Epoch 1/1, Batch 1/2500, Loss: 10.9140, LR: 0.00e+00
Epoch 1/1, Batch 2/2500, Loss: 10.9235, LR: 4.00e-06
Epoch 1/1, Batch 3/2500, Loss: 10.9069, LR: 8.00e-06
Epoch 1/1, Batch 4/2500, Loss: 10.8789, LR: 1.20e-05
Epoch 1/1, Batch 5/2500, Loss: 10.8397, LR: 1.60e-05
Epoch 1/1, Batch 6/2500, Loss: 10.7988, LR: 2.00e-05
Epoch 1/1, Batch 7/2500, Loss: 10.7464, LR: 2.40e-05
Epoch 1/1, Batch 8/2500, Loss: 10.6991, LR: 2.80e-05
Epoch 1/1, Batch 9/2500, Loss: 10.6438, LR: 3.20e-05
Epoch 1/1, Batch 10/2500, Loss: 10.5925, LR: 3.60e-05
Epoch 1/1, Batch 11/2500, Loss: 10.5416, LR: 4.00e-05
Epoch 1/1, Batch 12/2500, Loss: 10.4928, LR: 4.40e-05
Epoch 1/1, Batch 13/2500, Loss: 10.4369, LR: 4.80e-05
Epoch 1/1, Batch 14/2500, Loss: 10.3948, LR: 5.20e-05
Epoch 1/1, Batch 15/2500, Loss: 10.3521, LR: 5.60e-05
Epoch 1/1, Batch 16/2500, Loss: 10.3064, LR: 6.00e-05
Epoch 1/1, Batch 17/2500, Loss: 10.2643, LR: 6.40e-05
Epoch 1/1, Batch 18/2500, Loss: 10.2298, LR: 6.80e-05
Epoch 1/1, Batch 19/2500, Loss: 10.19

In [16]:
# Load the trained model and generate some recipes.
loaded_trainer = miniGPT.load(str(output_dir), map_location=device)

# Generate a few recipe samples with different prompts.
prompts = [
    "Ingredients: flour butter sugar",
    "Instructions: preheat oven",
    "Recipe for chocolate",
]

print("=" * 60)
print("GENERATED RECIPES")
print("=" * 60)

for prompt in prompts:
    print(f"\nPrompt: {prompt}")
    print("-" * 60)
    generated = loaded_trainer.generate(prompt, max_length=300, temperature=0.9, top_p=0.95)
    print(generated)
    print()

GENERATED RECIPES

Prompt: Ingredients: flour butter sugar
------------------------------------------------------------
Ingredients: flour butter sugar
- 1 c butter, melted
- 1/2 tsp baking powder
- 1 tsp baking soda
- 1/3 c brown sugar
- 1/3 c milk
- 1 1/2 c milk
- 2 c. fresh cranberries
- 1 tsp. cinnamon

Directions:
- Heat oven to 400°.
- Combine flour and butter.
- Add eggs and salt in a medium bowl.
- Blend well.
- Pour batter into balls and spread.
- Bake for 25 to 30-6 minutes.
- Add 1/2 cup sugar, vanilla, vanilla and milk.
- Pour into greased cookie sheet.
- Bake at 350° for 45 minutes or until set.
- Makes 4 servings.
- Makes 40 servings.


Prompt: Instructions: preheat oven
------------------------------------------------------------
Instructions: preheat oven

Ingredients:
- 2-1/2 tablespoons butter
- 2 tablespoons unsalted butter
- 2 1 tablespoon Worcestershire sauce
- 1 tablespoon brown sugar
- 2 garlic cloves, minced
- 1/2 teaspoon minced fresh or 1/4 teaspoon celery see